# Mini-projet : la courbe du CO₂ atmosphérique

| Module | Durée | Prérequis |
|---|---|---|
| M6 – Orthogonalité et moindres carrés | 150 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Depuis 1958, l'observatoire de **Mauna Loa**, à Hawaï, mesure en continu la concentration de dioxyde de carbone dans l'atmosphère. Cette série, appelée « courbe de Keeling » du nom du chercheur qui l'a lancée, est l'une des plus célèbres de la science du climat. Le fichier `donnees/co2_mauna_loa.csv` contient les moyennes mensuelles, en parties par million (ppm), de mars 1958 à août 2026. Il provient de la NOAA (agence américaine d'observation océanique et atmosphérique) et appartient au domaine public.

Une journaliste scientifique prépare un article et vous pose trois questions :

1. À quel rythme le CO₂ augmente-t-il, et ce rythme change-t-il ?
2. Pourquoi la courbe ondule-t-elle chaque année ?
3. Si la tendance actuelle se poursuit, **en quelle année** la concentration dépassera-t-elle 450 ppm, puis 500 ppm ? Avec quelle **incertitude** ?

**Activité** : Précisez ces questions. Que veut dire « si la tendance se poursuit » ? Sur quelles hypothèses repose toute prévision de ce type ? Quelle forme de réponse serait honnête : une année, ou un intervalle ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

co2 = pd.read_csv("donnees/co2_mauna_loa.csv")
t = co2["annee_decimale"].to_numpy()
c = co2["co2_ppm"].to_numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, c, color=LIOUAI["navy"], lw=0.8)
ax.set_xlabel("année"); ax.set_ylabel("CO₂ (ppm)"); ax.grid(alpha=0.3)
ax.set_title("Concentration de CO₂ à Mauna Loa, moyennes mensuelles")
plt.tight_layout(); plt.show()
co2.tail()

In [ ]:
pari_annee_450 = 2040      # (modifiez)
pari_annee_500 = 2070      # (modifiez)

## ③ Abstraire : une tendance et une saison, en colonnes

On décompose la concentration en trois parties :

$$
c(t) = \underbrace{\text{tendance}(t)}_{\text{lente}} + \underbrace{\text{saison}(t)}_{\text{périodique, 1 an}} + \underbrace{\text{reste}(t)}_{\text{ce qu'on n'explique pas}} .
$$

- Pour la **tendance**, on essaie des polynômes de degré 1, 2 ou 3 en $s = (t - 2000)/10$ (le temps en décennies, centré : module M5).
- Pour la **saison**, une fonction de période 1 an est bien approchée par quelques sinusoïdes : $\cos(2\pi t)$, $\sin(2\pi t)$, et leurs « harmoniques » $\cos(4\pi t)$, $\sin(4\pi t)$.

Le modèle complet est une combinaison linéaire de ces colonnes : les moindres carrés s'appliquent directement. Par exemple, pour une tendance de degré 2 et deux harmoniques,

$$
c(t) \approx x_0 + x_1 s + x_2 s^2 + x_3\cos(2\pi t) + x_4\sin(2\pi t) + x_5\cos(4\pi t) + x_6\sin(4\pi t).
$$

**Valider.** Pour choisir entre les modèles, on reprend la méthode du notebook 6.4 : on ajuste sur les données jusqu'en 2010, et l'on mesure l'erreur de **prévision** sur 2011-2026.

**Activité** : Pourquoi faut-il les **deux** colonnes $\cos(2\pi t)$ et $\sin(2\pi t)$, et pas seulement la première ? Indication : $a\cos(2\pi t) + b\sin(2\pi t) = R\cos(2\pi(t - t_0))$ ; que représentent $R$ et $t_0$ ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les colonnes du modèle

In [ ]:
def colonnes(t, degre=2, harmoniques=2):
    s = (t - 2000) / 10
    cols = [s ** k for k in range(degre + 1)]
    for k in range(1, harmoniques + 1):
        cols += [np.cos(2 * np.pi * k * t), np.sin(2 * np.pi * k * t)]
    return np.column_stack(cols)

def ajuster(t, c, degre=2, harmoniques=2):
    coef, *_ = np.linalg.lstsq(colonnes(t, degre, harmoniques), c, rcond=None)
    return coef

def prevoir(coef, t, degre=2, harmoniques=2):
    return colonnes(t, degre, harmoniques) @ coef

def reqm(y, y_prevu):
    return np.sqrt(np.mean((y - y_prevu) ** 2))

### Choisir la tendance : apprendre jusqu'en 2010, prévoir ensuite

In [ ]:
app = t < 2011
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(t, c, color=LIOUAI["gris"], lw=0.8, label="mesures")
ax.axvline(2011, color=LIOUAI["navy"], ls=":", lw=1)
for degre, couleur in [(1, LIOUAI["jaune"]), (2, LIOUAI["vert"]), (3, LIOUAI["navy"])]:
    coef = ajuster(t[app], c[app], degre)
    e_app = reqm(c[app], prevoir(coef, t[app], degre))
    e_test = reqm(c[~app], prevoir(coef, t[~app], degre))
    print(f"tendance de degré {degre} : REQM apprentissage {e_app:5.2f} ppm   prévision 2011-2026 {e_test:5.2f} ppm")
    ax.plot(t[~app], prevoir(coef, t[~app], degre), color=couleur, lw=1.2, label=f"prévision, degré {degre}")
ax.set_xlim(1995, 2027); ax.set_ylim(355, 440)
ax.set_xlabel("année"); ax.set_ylabel("CO₂ (ppm)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Modèles ajustés jusqu'en 2010, comparés aux mesures suivantes")
plt.tight_layout(); plt.show()

**Activité** : Quel degré retenez-vous ? Le degré 3 ajuste mieux le passé ; pourquoi prévoit-il moins bien ?

Indiquez vos réponses et justifiez-les.

### Le modèle retenu, sur toute la série

In [ ]:
coef = ajuster(t, c)
residus = c - prevoir(coef, t)
x0, x1, x2, a1, b1, a2, b2 = coef

annees = np.array([1960, 1980, 2000, 2020, 2026])
s = (annees - 2000) / 10
croissance = (x1 + 2 * x2 * s) / 10          # dérivée de la tendance, en ppm par an
for an, v in zip(annees, croissance):
    print(f"Croissance en {an} : {v:.2f} ppm par an")

amplitude = np.hypot(a1, b1)
print(f"\nSaison : amplitude du premier harmonique {amplitude:.2f} ppm, écart type des résidus {residus.std():.2f} ppm")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
mois = np.linspace(0, 1, 200)
saison = a1 * np.cos(2 * np.pi * mois) + b1 * np.sin(2 * np.pi * mois) + a2 * np.cos(4 * np.pi * mois) + b2 * np.sin(4 * np.pi * mois)
axes[0].plot(mois * 12 + 1, saison, color=LIOUAI["vert"], lw=2)
axes[0].set_xticks(range(1, 13), list("JFMAMJJASOND")); axes[0].axhline(0, color=LIOUAI["gris"])
axes[0].set_title("La partie saisonnière (ppm)")
axes[1].plot(t, residus, color=LIOUAI["navy"], lw=0.7); axes[1].axhline(0, color=LIOUAI["gris"])
axes[1].set_title("Les résidus au fil du temps (ppm)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
noms_mois = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août",
             "septembre", "octobre", "novembre", "décembre"]
print("Maximum saisonnier en", noms_mois[int(mois[np.argmax(saison)] * 12)],
      "; minimum en", noms_mois[int(mois[np.argmin(saison)] * 12)])

**Activité** : Les résidus ressemblent-ils à un bruit sans forme ? Que voyez-vous ? Une piste : certaines années, comme 1998 ou 2016, ont connu un fort phénomène **El Niño**, qui modifie les échanges de carbone entre océan, végétation et atmosphère.

Indiquez vos réponses et justifiez-les.

### Corroborer : comparer à la série désaisonnalisée de la NOAA

La NOAA publie sa propre version de la série, sans la saison. Notre tendance doit en être proche.

In [ ]:
tendance = prevoir(np.r_[coef[:3], np.zeros(4)], t)
ecart = co2["co2_desaisonnalise_noaa"].to_numpy() - tendance
print(f"Écart entre notre tendance et la série désaisonnalisée de la NOAA : moyenne {ecart.mean():+.2f} ppm, "
      f"écart type {ecart.std():.2f} ppm")

### Prévoir, avec une incertitude

Une seule année ne suffit pas comme réponse. On fait varier les choix raisonnables et l'on regarde comment la prévision bouge :

- la **forme** de la tendance (degré 2 ou 3) ;
- la **période** utilisée pour ajuster (les 20, 30 ou 40 dernières années, ou toute la série).

L'écart entre ces prévisions donne une idée, minimale, de l'incertitude liée aux choix du modèle.

In [ ]:
futur = np.arange(2026.7, 2080, 1 / 12)

def annee_franchissement(coef, seuil, degre):
    tendance_future = prevoir(np.r_[coef[:degre + 1], np.zeros(4)], futur, degre)
    au_dessus = tendance_future > seuil
    return futur[np.argmax(au_dessus)] if au_dessus.any() else np.nan

scenarios = []
fig, ax = plt.subplots(figsize=(10, 4.3))
ax.plot(t, c, color=LIOUAI["gris"], lw=0.6)
for degre in (2, 3):
    for debut in (1958, 1986, 1996, 2006):
        m = t >= debut
        cf = ajuster(t[m], c[m], degre)
        scenarios.append({"degré": degre, "depuis": debut,
                          "450 ppm": annee_franchissement(cf, 450, degre),
                          "500 ppm": annee_franchissement(cf, 500, degre)})
        ax.plot(futur, prevoir(np.r_[cf[:degre + 1], np.zeros(4)], futur, degre), lw=1,
                color=LIOUAI["vert"] if degre == 2 else LIOUAI["navy"], alpha=0.8)
for seuil in (450, 500):
    ax.axhline(seuil, color=LIOUAI["jaune"], ls="--", lw=1)
ax.set_xlim(1990, 2075); ax.set_ylim(340, 560); ax.grid(alpha=0.3)
ax.set_xlabel("année"); ax.set_ylabel("CO₂, tendance (ppm)")
ax.set_title("Huit prolongements de la tendance (vert : degré 2, bleu foncé : degré 3)")
plt.tight_layout(); plt.show()

scenarios = pd.DataFrame(scenarios).round(1)
print(scenarios.fillna("non atteint avant 2080").to_string(index=False))
for seuil in ("450 ppm", "500 ppm"):
    v = scenarios[seuil]
    print(f"\n{seuil} : entre {v.min():.0f} et {v.max():.0f}"
          + (f"  ({v.isna().sum()} scénario(s) ne l'atteignent pas avant 2080)" if v.isna().any() else ""), end="")
print()

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : 450 ppm en {pari_annee_450}, 500 ppm en {pari_annee_500}")

**Activité** :

1. Répondez aux trois questions de la journaliste, en donnant pour la troisième un intervalle d'années. Que pensez-vous du scénario de degré 3 ajusté sur les seules années récentes, dont la courbe s'infléchit ?
2. La croissance annuelle augmente-t-elle ? Que dit ce constat sur les émissions humaines ?
3. Expliquez l'ondulation annuelle. Indication : la plus grande partie des terres et de la végétation se trouve dans l'hémisphère Nord ; que se passe-t-il au printemps et en été ?
4. **Au-delà de l'étude.** Toutes nos prévisions supposent que les émissions continuent d'évoluer comme par le passé. Formulez une hypothèse sur ce qui pourrait rendre la prévision trop pessimiste, puis trop optimiste. Quelles données faudrait-il pour la tester ?

Indiquez vos réponses et justifiez-les.

### Le rapport

Rédigez pour la journaliste un rapport d'une page, structuré ainsi :

1. **La question**, reformulée en une phrase.
2. **Les données** : origine, période, ce qu'elles mesurent.
3. **La méthode**, sans formule : tendance et saison, validation sur 2011-2026.
4. **Les résultats**, avec un graphique et des intervalles plutôt que des valeurs isolées.
5. **Les limites** : extrapolation, hypothèses sur les émissions, incertitude sous-estimée.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Une incertitude sous-estimée.** L'intervalle obtenu ne tient compte que de quelques choix de modélisation. Il ignore les changements de politique climatique, les éruptions volcaniques, les rétroactions du climat sur la végétation et les océans. Faut-il l'élargir ? Comment le dire honnêtement ?
- **Des résidus corrélés.** Les résidus d'un mois ressemblent à ceux du mois précédent. Pourquoi est-ce gênant pour juger de la précision des coefficients ?
- **Généraliser : rejouer le passé.** Écrivez une fonction `rejouer(annee_coupure, degre, horizon=10)` qui ajuste le modèle sur les données antérieures à `annee_coupure` et renvoie l'erreur moyenne de prévision sur les `horizon` années suivantes. Appliquez-la pour des coupures de 1980 à 2015 : les modèles de degré 1, 2 et 3 auraient-ils bien prévu la décennie suivante ?

In [ ]:
def rejouer(annee_coupure, degre, horizon=10):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def rejouer(annee_coupure, degre, horizon=10):
    avant = t < annee_coupure
    apres = (t >= annee_coupure) & (t < annee_coupure + horizon)
    cf = ajuster(t[avant], c[avant], degre)
    return reqm(c[apres], prevoir(cf, t[apres], degre))

coupures = range(1980, 2016, 5)
fig, ax = plt.subplots(figsize=(8, 3.6))
for degre, couleur in [(1, LIOUAI["jaune"]), (2, LIOUAI["vert"]), (3, LIOUAI["navy"])]:
    erreurs = [rejouer(a, degre) for a in coupures]
    ax.plot(list(coupures), erreurs, "o-", color=couleur, label=f"degré {degre}")
    print(f"degré {degre} : erreur moyenne de prévision à 10 ans {np.mean(erreurs):.2f} ppm")
ax.set_xlabel("année de coupure"); ax.set_ylabel("REQM sur les 10 ans suivants (ppm)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Sentir la fiabilité d'un modèle": 1,
    "Corroborer ses résultats": 1,
    "Formuler une hypothèse au-delà de l'étude": 1,
    "Structurer un rapport ou une présentation": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M6-projet-co2", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")